# 开源模型架构一览

2026年的前沿模型与GPT-2是同一个家族，仅存在5到6处变更：
- RMSNorm 还是 LayerNorm
- SwiGLU 还是 GELU
- RoPE 还是可学习的位置编码
- GQA,MLA 还是 MHA
- MoE

## 基本概念

### 不变的核心

所有自回归的模型以下内容都是相似的：
- 词元嵌入矩阵
- N个解码块的堆叠
- 终层的归一化和输出头
- 因果掩码，交叉熵损失

### 六个可变的旋钮

分别为：
- 归一化。 LayerNorm --> RMSNorm
- 位置编码。 绝对位置编码 --> RoPE(变体：YaRN， NTK)
- 激活函数。 GELU --> SwiGLU
- 注意力头。  MHA --> GQA --> MQA --> MLA
- 稠密还是稀疏。 Dense --> MoE
- 归一化位置。 Pre-norm 胜出，Post-norm消失。

### RMSNorm

相对于LayerNorm 去掉了均值处理，且没有偏置。

### RoPE

相对于绝对位置编码，可以支持可变上下文窗口长度。

### SwiGLU

```
SwiGLU = (xW1) * sigmoid(xW1) * xV
```
维度放大系数大致为 `ff_dim = (2/3)*4*hidden = 8/3 hidden`

### 注意力头

MQA。  只有一组KV，多个头在相同的KV上查询计算，但是对于性能有损耗。

GQA。  每X个Q头共用一组KV。

MLA。  将KV压入更低维度的潜向量。在需要计算时再投影回来。

### MoE

稠密模型每次运算所有的权重都参与，MoE只有路由到的几个专家才参与。主要问题在于，所有的专家都需要加载到显存中以备使用，以及专家负载、路由器的均衡问题。

### Pre-norm Stays

原始的模型使用后归一化，存在难训练的问题。目前几乎所有的模型都适用前归一化。

In [ ]:

CONFIGS = {
    "gpt2-small": {
        "hidden_size": 768, "intermediate_size": 3072,
        "num_hidden_layers": 12, "num_attention_heads": 12,
        "num_key_value_heads": 12, "vocab_size": 50257,
        "max_position_embeddings": 1024,
        "activation": "gelu", "norm": "layernorm",
        "position": "learned", "moe": False,
    },
    "mistral-7b": {
        "hidden_size": 4096, "intermediate_size": 14336,
        "num_hidden_layers": 32, "num_attention_heads": 32,
        "num_key_value_heads": 8, "vocab_size": 32000,
        "max_position_embeddings": 32768,
        "activation": "swiglu", "norm": "rmsnorm",
        "position": "rope", "moe": False,
    },
    "llama3-8b": {
        "hidden_size": 4096, "intermediate_size": 14336,
        "num_hidden_layers": 32, "num_attention_heads": 32,
        "num_key_value_heads": 8, "vocab_size": 128256,
        "max_position_embeddings": 131072,
        "activation": "swiglu", "norm": "rmsnorm",
        "position": "rope", "moe": False,
    },
    "llama3-70b": {
        "hidden_size": 8192, "intermediate_size": 28672,
        "num_hidden_layers": 80, "num_attention_heads": 64,
        "num_key_value_heads": 8, "vocab_size": 128256,
        "max_position_embeddings": 131072,
        "activation": "swiglu", "norm": "rmsnorm",
        "position": "rope", "moe": False,
    },
    "mixtral-8x7b": {
        "hidden_size": 4096, "intermediate_size": 14336,
        "num_hidden_layers": 32, "num_attention_heads": 32,
        "num_key_value_heads": 8, "vocab_size": 32000,
        "max_position_embeddings": 32768,
        "activation": "swiglu", "norm": "rmsnorm",
        "position": "rope",
        "moe": True, "num_experts": 8, "experts_per_token": 2,
    },
    "qwen2.5-72b": {
        "hidden_size": 8192, "intermediate_size": 29568,
        "num_hidden_layers": 80, "num_attention_heads": 64,
        "num_key_value_heads": 8, "vocab_size": 152064,
        "max_position_embeddings": 131072,
        "activation": "swiglu", "norm": "rmsnorm",
        "position": "rope-yarn", "moe": False,
    },
    "deepseek-v3": {
        "hidden_size": 7168, "intermediate_size": 18432,
        "moe_intermediate_size": 2048,
        "num_hidden_layers": 61, "first_dense_layers": 3,
        "num_attention_heads": 128,
        "num_key_value_heads": 128, "vocab_size": 129280,
        "max_position_embeddings": 131072,
        "activation": "swiglu", "norm": "rmsnorm",
        "position": "rope",
        "moe": True, "num_experts": 256, "experts_per_token": 8,
        "shared_experts": 1,
        "attention": "mla", "kv_lora_rank": 512,
    },
}